<a href="https://colab.research.google.com/github/Camilalozano/WebscrapingMinutasSecop/blob/main/Scrape_sewcop_fiicha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalar dependencias

In [1]:
!apt-get update -y
!apt-get install -y wget gnupg unzip

!wget -q -O - https://dl.google.com/linux/linux_signing_key.pub | apt-key add -
!bash -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" > /etc/apt/sources.list.d/google-chrome.list'

!apt-get update -y
!apt-get install -y google-chrome-stable
!google-chrome --version

!pip install -q -U selenium pandas beautifulsoup4 lxml

Hit:1 http://dl.google.com/linux/chrome/deb stable InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,301 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,780 kB]
Fetched 5,210 kB in 3s (1,578 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.il

In [2]:
from google.colab import files

uploaded = files.upload()
EXCEL_PATH = list(uploaded.keys())[0]  # ✅ usa el archivo que acabas de subir
print("📄 Voy a leer:", EXCEL_PATH)


Saving URL_Solo 2.xlsx to URL_Solo 2 (3).xlsx
📄 Voy a leer: URL_Solo 2 (3).xlsx


In [3]:
def run():
    print("📄 Leyendo Excel:", EXCEL_PATH)   # ✅ debug
    urls = load_urls_from_excel(EXCEL_PATH, EXCEL_COL)
    print(f"🔗 URLs cargadas: {len(urls)}")

Habilitar navegador “visible” (NoVNC) para poder hacer click en reCAPTCHA

In [4]:
!apt-get install -y xvfb x11vnc novnc websockify
!mkdir -p /root/.vnc

# Password opcional (puedes dejarlo sin password si estás sola)
!x11vnc -storepasswd 1234 /root/.vnc/passwd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
novnc is already the newest version (1:1.0.0-5).
websockify is already the newest version (0.10.0+dfsg1-2build1).
x11vnc is already the newest version (0.9.16-8).
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
The following packages were automatically installed and are no longer required:
  apparmor libfuse3-3 snapd squashfs-tools systemd-hwe-hwdb udev
Use 'apt autoremove' to remove them.
0 upgraded, 0 newly installed, 0 to remove and 112 not upgraded.
stored passwd in file: /root/.vnc/passwd


Arrancar escritorio virtual + VNC + NoVNC

In [5]:
import os, subprocess, time

# Display virtual
subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1280x720x24", "-ac"])
os.environ["DISPLAY"] = ":99"

# VNC server
subprocess.Popen(["x11vnc", "-display", ":99", "-forever", "-passwdfile", "/root/.vnc/passwd", "-rfbport", "5900"])

# NoVNC web
subprocess.Popen(["websockify", "--web=/usr/share/novnc/", "6080", "localhost:5900"])

time.sleep(2)
print("✅ NoVNC corriendo en el puerto 6080")

✅ NoVNC corriendo en el puerto 6080


Abrir el visor NoVNC (para ver Chrome)

In [6]:
from google.colab import output
print("Abre esto en una pestaña nueva:")
output.eval_js("window.open('https://localhost:6080/vnc.html?autoconnect=true&password=1234','_blank')")

Abre esto en una pestaña nueva:


Subir tu Excel (si no está ya cargado)

In [7]:
!which chromium
!which chromium-browser
!which google-chrome
!which chromedriver
!chromium --version || true
!chromium-browser --version || true
!chromedriver --version || true
!ls -la /usr/lib/chromium-browser/ || true

/usr/bin/google-chrome
/bin/bash: line 1: chromium: command not found
/bin/bash: line 1: chromium-browser: command not found
/bin/bash: line 1: chromedriver: command not found
ls: cannot access '/usr/lib/chromium-browser/': No such file or directory


: leer Excel → abrir URL → si hay reCAPTCHA, tú lo resuelves en la ventana NoVNC → extraer ficha → guardar CSV/JSONL/SQLite.

In [8]:
import ast
import json
import re
import sqlite3
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait


# =========================
# Config
# =========================
EXCEL_COL = "urlproceso (Contratos_electronicos)_SecopDatosAbiertos"

OUT_DIR = Path("out_secop")
OUT_DIR.mkdir(exist_ok=True)

DB_PATH = OUT_DIR / "secop_fichas.sqlite"
CSV_PATH = OUT_DIR / "secop_fichas.csv"
JSONL_PATH = OUT_DIR / "secop_fichas.jsonl"

PAGE_LOAD_TIMEOUT = 60
WAIT_TIMEOUT = 25


def now_iso() -> str:
    return datetime.utcnow().isoformat(timespec="seconds") + "Z"


def normalize_key(k: str) -> str:
    return re.sub(r"\s+", " ", (k or "").strip()).strip(":").strip()


def normalize_val(v: str) -> str:
    return re.sub(r"\s+", " ", (v or "").strip())


def parse_url_cell(cell: str) -> str | None:
    if not isinstance(cell, str) or not cell.strip():
        return None
    s = cell.strip()

    if s.startswith("http"):
        return s

    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, dict) and "url" in obj:
            return str(obj["url"]).strip()
    except Exception:
        pass

    m = re.search(r"(https?://\S+)", s)
    return m.group(1) if m else None


def load_urls_from_excel(path: str, col: str) -> list[str]:
    df = pd.read_excel(path)
    if col not in df.columns:
        raise ValueError(f"❌ No encuentro la columna '{col}'. Columnas: {list(df.columns)}")

    urls = []
    for x in df[col].dropna().tolist():
        u = parse_url_cell(x)
        if u:
            urls.append(u)

    seen = set()
    out = []
    for u in urls:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


from selenium.webdriver.chrome.service import Service

import shutil
from selenium.webdriver.chrome.service import Service
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

import os
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

def make_driver():
    chrome_opts = Options()
    chrome_opts.binary_location = "/usr/bin/google-chrome"

    chrome_opts.add_argument("--no-sandbox")
    chrome_opts.add_argument("--disable-dev-shm-usage")
    chrome_opts.add_argument("--window-size=1280,720")
    chrome_opts.add_argument("--disable-gpu")
    chrome_opts.add_argument("--remote-debugging-port=9222")

    # Si NO vas a usar NoVNC (captcha manual), deja headless.
    # Si SÍ vas a resolver captcha manual, COMENTA esta línea.
    # chrome_opts.add_argument("--headless=new")

    driver = webdriver.Chrome(options=chrome_opts)
    driver.set_page_load_timeout(60)
    return driver


def is_recaptcha_page(driver: webdriver.Chrome) -> bool:
    url = (driver.current_url or "").lower()
    html = (driver.page_source or "").lower()

    if "googlerecaptcha" in url:
        return True
    if "recaptcha" in html and "i'm not a robot" in html:
        return True
    try:
        if driver.find_elements(By.CSS_SELECTOR, "iframe[src*='recaptcha']"):
            return True
    except Exception:
        pass
    return False


def wait_for_manual_recaptcha(driver: webdriver.Chrome) -> None:
    print("🧩 Detecté reCAPTCHA. Resuélvelo manualmente en la ventana NoVNC (Chrome visible).")
    input("✅ Cuando termines, presiona ENTER aquí para continuar...")
    time.sleep(2)


def extract_kv_pairs_generic(html: str) -> dict:
    soup = BeautifulSoup(html, "lxml")
    data: dict[str, str] = {}

    # dt/dd
    for dl in soup.select("dl"):
        dts = dl.select("dt")
        dds = dl.select("dd")
        if len(dts) == len(dds) and len(dts) > 0:
            for dt, dd in zip(dts, dds):
                k = normalize_key(dt.get_text(" ", strip=True))
                v = normalize_val(dd.get_text(" ", strip=True))
                if k and v and k not in data:
                    data[k] = v

    # tables
    for table in soup.select("table"):
        for tr in table.select("tr"):
            cells = tr.find_all(["th", "td"])
            if len(cells) == 2:
                k = normalize_key(cells[0].get_text(" ", strip=True))
                v = normalize_val(cells[1].get_text(" ", strip=True))
                if k and v and k not in data:
                    data[k] = v

    # "Clave: Valor" heuristic
    for row in soup.select(".row, .form-group, .field, .item"):
        text = row.get_text(" ", strip=True)
        if ":" in text:
            parts = text.split(":", 1)
            k = normalize_key(parts[0])
            v = normalize_val(parts[1])
            if 2 <= len(k) <= 80 and v and k not in data:
                data[k] = v

    return data


def extract_notice_uid(url: str) -> str | None:
    m = re.search(r"noticeUID=([^&]+)", url)
    return m.group(1) if m else None


def init_db(db_path: Path) -> None:
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS fichas (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            scraped_at TEXT,
            url TEXT,
            notice_uid TEXT,
            status TEXT,
            fields_json TEXT,
            raw_html_path TEXT
        )
    """)
    con.commit()
    con.close()


def save_one(db_path: Path, record: dict) -> None:
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("""
        INSERT INTO fichas (scraped_at, url, notice_uid, status, fields_json, raw_html_path)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        record.get("scraped_at"),
        record.get("url"),
        record.get("notice_uid"),
        record.get("status"),
        json.dumps(record.get("fields", {}), ensure_ascii=False),
        record.get("raw_html_path"),
    ))
    con.commit()
    con.close()


def append_jsonl(path: Path, record: dict) -> None:
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def run():
    urls = load_urls_from_excel(EXCEL_PATH, EXCEL_COL)
    print(f"🔗 URLs cargadas: {len(urls)}")

    init_db(DB_PATH)

    driver = make_driver()
    wait = WebDriverWait(driver, WAIT_TIMEOUT)

    all_rows = []
    try:
        for i, url in enumerate(urls, start=1):
            print(f"\n➡️ ({i}/{len(urls)}) Abriendo: {url}")

            record = {
                "scraped_at": now_iso(),
                "url": url,
                "notice_uid": extract_notice_uid(url),
                "status": "init",
                "fields": {},
                "raw_html_path": None,
            }

            try:
                driver.get(url)

                if is_recaptcha_page(driver):
                    record["status"] = "captcha"
                    wait_for_manual_recaptcha(driver)

                wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
                time.sleep(1.2)

                html = driver.page_source

                html_path = OUT_DIR / f"raw_{i:04d}.html"
                html_path.write_text(html, encoding="utf-8")
                record["raw_html_path"] = str(html_path)

                fields = extract_kv_pairs_generic(html)
                record["fields"] = fields
                record["status"] = "ok" if fields else "ok_empty"

            except Exception as e:
                record["status"] = f"error:{type(e).__name__}"
                record["error"] = str(e)

            save_one(DB_PATH, record)
            append_jsonl(JSONL_PATH, record)

            flat = {
                "scraped_at": record["scraped_at"],
                "url": record["url"],
                "notice_uid": record["notice_uid"],
                "status": record["status"]
            }
            for k, v in (record.get("fields") or {}).items():
                flat[k] = v
            all_rows.append(flat)

            time.sleep(1.0)

    finally:
        driver.quit()

    df_out = pd.DataFrame(all_rows)
    df_out.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"\n💾 Listo. CSV: {CSV_PATH} | JSONL: {JSONL_PATH} | SQLite: {DB_PATH}")


run()

🔗 URLs cargadas: 2

➡️ (1/2) Abriendo: https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7301080&isFromPublicArea=True&isModal=true&asPopupView=true


/tmp/ipykernel_58502/1192344487.py:35: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().isoformat(timespec="seconds") + "Z"


🧩 Detecté reCAPTCHA. Resuélvelo manualmente en la ventana NoVNC (Chrome visible).
✅ Cuando termines, presiona ENTER aquí para continuar...

➡️ (2/2) Abriendo: https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.7305903&isFromPublicArea=True&isModal=true&asPopupView=true
🧩 Detecté reCAPTCHA. Resuélvelo manualmente en la ventana NoVNC (Chrome visible).
✅ Cuando termines, presiona ENTER aquí para continuar...

💾 Listo. CSV: out_secop/secop_fichas.csv | JSONL: out_secop/secop_fichas.jsonl | SQLite: out_secop/secop_fichas.sqlite


Descargar resultados

In [9]:
from google.colab import files
files.download("out_secop/secop_fichas.csv")
files.download("out_secop/secop_fichas.jsonl")
files.download("out_secop/secop_fichas.sqlite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>